In [0]:
-- merge CSVs files 
create table google_drive.restaurant_all
as
select * from google_drive.restaurant_1
union all
select * from google_drive.restaurant_2
union all
select * from google_drive.restaurant_3
union all
select * from google_drive.restaurant_4
union all 
select * from google_drive.restaurant_5
union all
select * from google_drive.restaurant_6
union all
select * from google_drive.restaurant_7;

select * from google_drive.restaurant_all;
-- merge JSONs files 
create table google_drive.restaurant_all_json
as
select * from google_drive.restaurant_json_1
union all
select * from google_drive.restaurant_json_2;

select * from google_drive.restaurant_all_json;

-- merge CSVs & JSONs files (excluding _line and _fivetran_synced columns)
create or replace table google_drive.restaurant_table
as
select 
  order_id,
  order_date,
  hour,
  category,
  item_name,
  price,
  quantity,
  discount,
  total_amount,
  branch,
  payment_method,
  order_type,
  customer_id,
  rating,
  is_weekend
from google_drive.restaurant_all
union all
select 
  order_id,
  order_date,
  hour,
  category,
  item_name,
  price,
  quantity,
  discount,
  total_amount,
  branch,
  payment_method,
  order_type,
  customer_id,
  rating,
  is_weekend
from google_drive.restaurant_all_json;

select * from google_drive.restaurant_table;


-- add columns 
alter table google_drive.restaurant_table
add columns ( 
  year int,
  month int,
  day_name string);


update  google_drive.restaurant_table
set 
    year = year(order_date),
    month = month(order_date),
    day_name = date_format(order_date,"EEEE");





--- measures

select count(distinct order_id)
from google_drive.restaurant_table;

select count(distinct customer_id)
from google_drive.restaurant_table;


select count(distinct category)
from google_drive.restaurant_table;


select count(distinct branch)
from google_drive.restaurant_table;

select count(distinct item_name)
from google_drive.restaurant_table;


select sum(total_amount)
from google_drive.restaurant_table;

select avg(total_amount)
from google_drive.restaurant_table;

select count(distinct payment_method)
from google_drive.restaurant_table;

select count(distinct order_type)
from google_drive.restaurant_table;

-- total quantity 
select sum(quantity)
from google_drive.restaurant_table;



select avg(discount)
from google_drive.restaurant_table;



select avg(price)
from google_drive.restaurant_table;


-- retention rate
WITH customer_monthly_orders AS (
  SELECT 
    customer_id,
    DATE_TRUNC('month', order_date) as order_month,
    COUNT(DISTINCT order_id) as orders
  FROM google_drive.restaurant_table
  GROUP BY customer_id, DATE_TRUNC('month', order_date)
),
customer_retention AS (
  SELECT 
    curr.order_month,
    COUNT(DISTINCT curr.customer_id) as customers_this_month,
    COUNT(DISTINCT CASE 
      WHEN prev.customer_id IS NOT NULL 
      THEN curr.customer_id 
    END) as retained_customers
  FROM customer_monthly_orders curr
  LEFT JOIN customer_monthly_orders prev 
    ON curr.customer_id = prev.customer_id 
    AND prev.order_month = ADD_MONTHS(curr.order_month, -1)
  GROUP BY curr.order_month
)
SELECT 
  order_month,
  customers_this_month,
  retained_customers,
  ROUND(retained_customers * 100.0 / customers_this_month, 2) as retention_rate_pct
FROM customer_retention
ORDER BY order_month;




-- -ve values
select count(*)
from google_drive.restaurant_table
where price > 0 or total_amount > 0;

select sum(total_amount)
from google_drive.restaurant_table
where price > 0 or total_amount > 0;



select * from google_drive.restaurant_table